In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from lightgbm import LGBMClassifier

In [20]:
df = pd.read_csv("C:\\Users\\mkart\\OneDrive\\Desktop\\Ai-Fraud-Risk-Intelligence\\data\\financial_fraud_detection_dataset.csv")

In [21]:
print(df.shape)
print(df.columns)
print(df.head())

(5000000, 18)
Index(['transaction_id', 'timestamp', 'sender_account', 'receiver_account',
       'amount', 'transaction_type', 'merchant_category', 'location',
       'device_used', 'is_fraud', 'fraud_type', 'time_since_last_transaction',
       'spending_deviation_score', 'velocity_score', 'geo_anomaly_score',
       'payment_channel', 'ip_address', 'device_hash'],
      dtype='str')
  transaction_id                   timestamp sender_account receiver_account  \
0        T100000  2023-08-22T09:22:43.516168      ACC877572        ACC388389   
1        T100001  2023-08-04T01:58:02.606711      ACC895667        ACC944962   
2        T100002  2023-05-12T11:39:33.742963      ACC733052        ACC377370   
3        T100003  2023-10-10T06:04:43.195112      ACC996865        ACC344098   
4        T100004  2023-09-24T08:09:02.700162      ACC584714        ACC497887   

    amount transaction_type merchant_category location device_used  is_fraud  \
0   343.78       withdrawal         utilities    To

In [22]:
df_model = df.copy()

In [23]:
df_model = df_model.drop(
    columns=[
        'transaction_id',
        'timestamp',
        'sender_account',
        'receiver_account',
        'fraud_type',
        'ip_address',
        'device_hash'
    ]
)

In [24]:
df_model['time_missing_flag'] = (
    df_model['time_since_last_transaction']
    .isnull()
    .astype(int)
)

In [25]:
df_model['time_since_last_transaction'] = (
    df_model['time_since_last_transaction']
    .fillna(0)
)

In [26]:
df_model = pd.get_dummies(
    df_model,
    columns=[
        'transaction_type',
        'merchant_category',
        'location',
        'device_used',
        'payment_channel'
    ],
    drop_first=True
)

In [27]:
df_model = df_model.drop(
    columns=[
        'time_missing_flag',
        'fraud_type_not_fraud'
    ],
    errors='ignore'
)

In [28]:
print(df_model.shape)
print(df_model.dtypes.value_counts())
print(
    df_model.select_dtypes(
        include=['object']
    ).columns
)

(5000000, 29)
bool       24
float64     4
int64       1
Name: count, dtype: int64
Index([], dtype='str')


In [29]:
X = df_model.drop('is_fraud', axis=1)
y = df_model['is_fraud']

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [31]:
print(X_train.shape)
print(X_train.dtypes.value_counts())

(4000000, 28)
bool       23
float64     4
int64       1
Name: count, dtype: int64


In [32]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

lgbm.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 143642, number of negative: 3856358
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053424 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 933
[LightGBM] [Info] Number of data points in the train set: 4000000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [33]:
y_pred_lgbm = lgbm.predict(X_test)
y_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]

In [34]:
print(classification_report(y_test, y_pred_lgbm))
print(confusion_matrix(y_test, y_pred_lgbm))

print("Accuracy :", accuracy_score(y_test, y_pred_lgbm))
print("Precision:", precision_score(y_test, y_pred_lgbm))
print("Recall   :", recall_score(y_test, y_pred_lgbm))
print("F1 Score :", f1_score(y_test, y_pred_lgbm))

              precision    recall  f1-score   support

       False       1.00      0.20      0.34    964089
        True       0.04      0.98      0.08     35911

    accuracy                           0.23   1000000
   macro avg       0.52      0.59      0.21   1000000
weighted avg       0.96      0.23      0.33   1000000

[[195908 768181]
 [   789  35122]]
Accuracy : 0.23103
Precision: 0.04372198286325334
Recall   : 0.9780290161788867
F1 Score : 0.08370213080334694


In [35]:
print("Maximum Probability :", y_prob_lgbm.max())
print("Mean Probability    :", y_prob_lgbm.mean())

Maximum Probability : 0.6784823990677704
Mean Probability    : 0.4515402512758252


In [36]:
feature_importance_lgbm = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': lgbm.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance_lgbm.head(15))

                          Feature  Importance
2        spending_deviation_score         556
1     time_since_last_transaction         544
0                          amount         505
4               geo_anomaly_score         433
3                  velocity_score         265
27  payment_channel_wire_transfer          42
17              location_New York          39
22             device_used_mobile          39
26           payment_channel_card          37
23                device_used_pos          37
11   merchant_category_restaurant          34
8       merchant_category_grocery          34
20                 location_Tokyo          34
15                 location_Dubai          34
25            payment_channel_UPI          34


In [37]:
print(classification_report(y_test, y_pred_lgbm))
print(confusion_matrix(y_test, y_pred_lgbm))

              precision    recall  f1-score   support

       False       1.00      0.20      0.34    964089
        True       0.04      0.98      0.08     35911

    accuracy                           0.23   1000000
   macro avg       0.52      0.59      0.21   1000000
weighted avg       0.96      0.23      0.33   1000000

[[195908 768181]
 [   789  35122]]


In [43]:
txn_no = int(input("Enter transaction number: "))

if txn_no not in X.index:
    print("❌ Transaction not found.")

else:
    sample = X.loc[[txn_no]]
    row = sample.iloc[0]

    # -------------------------
    # Fraud Probability
    # -------------------------
    prob = lgbm.predict_proba(sample)[0][1]

    # -------------------------
    # Risk Classification
    # -------------------------
    amount = row['amount']

    if amount < 100:
        risk = "🟢 Low Risk"
        status = "NOT FLAGGED"
        recommendation = (
            "Low-value transaction (< ₹100). "
            "No fraud escalation required."
        )

    else:
        if prob >= 0.60:
            risk = "🔴 High Risk Fraud"
            status = "FLAGGED"
            recommendation = (
                "Temporarily hold transaction and investigate."
            )

        elif prob >= 0.30:
            risk = "🟡 Suspicious"
            status = "REVIEW REQUIRED"
            recommendation = (
                "Perform additional verification."
            )

        else:
            risk = "🟢 Low Risk"
            status = "NOT FLAGGED"
            recommendation = (
                "Approve transaction."
            )

    # -------------------------
    # Reasons
    # -------------------------
    reasons = []

    if row['amount'] > X['amount'].quantile(0.90):
        reasons.append("High transaction amount")

    if row['velocity_score'] > X['velocity_score'].quantile(0.90):
        reasons.append("High transaction velocity")

    if row['geo_anomaly_score'] > X['geo_anomaly_score'].quantile(0.90):
        reasons.append("Unusual geographical activity")

    if abs(row['spending_deviation_score']) > 1:
        reasons.append("Abnormal spending behaviour")

    # -------------------------
    # Transaction Location
    # -------------------------
    location_cols = [
        c for c in sample.columns
        if c.startswith('location_')
    ]

    transaction_location = "Reference Location"

    for col in location_cols:
        if row[col]:
            transaction_location = col.replace(
                'location_',
                ''
            )
            break

    customer_home_location = "Chennai"

    if customer_home_location != transaction_location:
        reasons.append(
            f"Transaction occurred from "
            f"{transaction_location}, "
            f"which differs from the "
            f"customer's usual location "
            f"({customer_home_location})"
        )

    # -------------------------
    # Report
    # -------------------------
    print("\n")
    print("=" * 55)
    print("       LIGHTGBM FRAUD ANALYSIS REPORT")
    print("=" * 55)

    print(f"Transaction Number : {txn_no}")
    print(f"Transaction Amount : ₹{amount:,.2f}")
    print(f"Fraud Probability  : {prob:.2%}")
    print(f"Risk Level         : {risk}")
    print(f"Status             : {status}")

    print(
        f"Customer Location  : "
        f"{customer_home_location}"
    )

    print(
        f"Transaction Location : "
        f"{transaction_location}"
    )

    print("\nWhy was this decision made?")

    if risk == "🟢 Low Risk":

        print(
            "• Overall fraud probability "
            "is low."
        )

        if reasons:
            print(
                "• Some unusual characteristics "
                "were observed, but they were "
                "not strong enough in combination "
                "to indicate fraud."
            )

    elif risk == "🟡 Suspicious":

        print(
            "• The transaction exhibits "
            "some risk indicators:"
        )

        for r in reasons:
            print("  -", r)

        print(
            "• Additional verification "
            "is recommended."
        )

    else:

        print(
            "• Multiple high-risk "
            "characteristics were detected:"
        )

        for r in reasons:
            print("  -", r)

        print(
            "• The combined evidence "
            "resulted in a high fraud "
            "probability."
        )

    print("\nRecommended Action:")
    print("•", recommendation)

    print("=" * 55)



       LIGHTGBM FRAUD ANALYSIS REPORT
Transaction Number : 3942363
Transaction Amount : ₹2,315.23
Fraud Probability  : 63.07%
Risk Level         : 🔴 High Risk Fraud
Status             : FLAGGED
Customer Location  : Chennai
Transaction Location : Tokyo

Why was this decision made?
• Multiple high-risk characteristics were detected:
  - High transaction amount
  - Transaction occurred from Tokyo, which differs from the customer's usual location (Chennai)
• The combined evidence resulted in a high fraud probability.

Recommended Action:
• Temporarily hold transaction and investigate.
